In [ ]:
import pandas as pd
import numpy as np
import ast

In [ ]:
# Load data
listings_path = 'data/listings.csv'
housing_path = 'data/bau522od5221_wohnungsbestand_zurich.csv'
rental_path = 'data/rental_prices.csv'

airbnb_df= pd.read_csv(listings_path)
housing_df = pd.read_csv(housing_path)
rental_df = pd.read_csv(rental_path)

print('Airbnb shape:', airbnb_df.shape)
print('Housing shape:', housing_df.shape)
print('Rental shape:', rental_df.shape)

In [ ]:
def overview(df, name="dataset"):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True)
    }).sort_values(["dtype", "missing_pct"], ascending=[True, False])

    print(f"\n=== {name} ===")
    print("Shape:", df.shape)
    display(summary)
    return summary

def clean_column_names(data):
    data = data.copy()
    data.columns = (
        data.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(r"[^\w]", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )
    return data

In [ ]:
airbnb_df = clean_column_names(airbnb_df)
housing_df = clean_column_names(housing_df)
rental_df = clean_column_names(rental_df)

overview(airbnb_df, "Airbnb")
overview(housing_df, "Housing")
overview(rental_df, "Rental")

airbnb_df.head()
airbnb_df.sample(5)
airbnb_df.info()

In [ ]:
def pct_to_float(x):
    if pd.isna(x):
        return np.nan
    return float(str(x).replace("%", "")) / 100

def price_to_float(x):
    if pd.isna(x):
        return np.nan
    return float(str(x).replace("$", "").replace(",", "").strip())

def tf_to_bool(x):
    if pd.isna(x):
        return np.nan
    if x == "t":
        return 1
    if x == "f":
        return 0
    return np.nan

def extract_country(location):
    if pd.isna(location):
        return np.nan
    parts = [p.strip() for p in str(location).split(",")]
    return parts[-1] if len(parts) > 0 else np.nan

def contains_zurich(location):
    if pd.isna(location):
        return 0
    s = str(location).lower()
    return int(("zurich" in s) or ("zürich" in s))

def parse_list_string(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if not isinstance(x, str):
        print("Unexpected type:", type(x), x)
        return []
    return ast.literal_eval(x)

# Aribnb

In [ ]:
reference_date = pd.Timestamp(airbnb_df["calendar_last_scraped"][0]) # 2025-09-29
df_airbnb_f = airbnb_df.copy()

# drop noisy columns
to_drop = [
    "name", "host_name", 
    "id", "listing_url", "scrape_id", "last_scraped", "source", "picture_url",
    "host_id", "host_url", "host_thumbnail_url", "host_picture_url",
    "calendar_last_scraped", "license", "has_availability", "calendar_updated", "host_neighbourhood", "neighbourhood"
]

df_airbnb_f = df_airbnb_f.drop(columns=[c for c in to_drop if c in df_airbnb_f.columns])
df_airbnb_f = df_airbnb_f[df_airbnb_f["price"].notna()] # drop all rows with missing price

# text columns - text length as proxy for effort/welcomeness?
text_cols = ["host_about", "description", "neighborhood_overview"]
for col in text_cols:
    if col in df_airbnb_f.columns:
        df_airbnb_f[f"{col}_len"] = (
            df_airbnb_f[col]
            .fillna("")
            .astype(str)
            .str.len()
        )
df_airbnb_f = df_airbnb_f.drop(columns=[c for c in text_cols if c in df_airbnb_f.columns])


# # # # dates
date_cols = ["host_since", "first_review", "last_review"]
for col in date_cols:
    df_airbnb_f[col] = pd.to_datetime(df_airbnb_f[col], errors="coerce")
# convert to dates since reference
df_airbnb_f["host_tenure_days"] = (reference_date - df_airbnb_f["host_since"]).dt.days
df_airbnb_f["days_since_first_review"] = (reference_date - df_airbnb_f["first_review"]).dt.days
df_airbnb_f["days_since_last_review"] = (reference_date - df_airbnb_f["last_review"]).dt.days
df_airbnb_f.drop(columns=date_cols, inplace=True) # remove date columns

# convert percents to float
for col in ["host_response_rate", "host_acceptance_rate"]:
    df_airbnb_f[col] = df_airbnb_f[col].apply(pct_to_float)

# convert price to float
df_airbnb_f["price"] = df_airbnb_f["price"].apply(price_to_float)

# convert tf to bool
bool_cols = [
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "has_availability",
    "instant_bookable"
]
for col in bool_cols:
    if col in df_airbnb_f.columns:
        df_airbnb_f[col] = df_airbnb_f[col].apply(tf_to_bool)

# host location derived
df_airbnb_f["host_country"] = df_airbnb_f["host_location"].apply(extract_country)
invalid_country_values = ["NY", "HI"] # probably New York, Hawaii
df_airbnb_f["host_country"] = df_airbnb_f["host_country"].replace(invalid_country_values, pd.NA)
df_airbnb_f["host_in_switzerland"] = df_airbnb_f["host_country"].eq("Switzerland").astype(float)
df_airbnb_f["host_in_zurich"] = df_airbnb_f["host_location"].apply(contains_zurich)
df_airbnb_f.drop(columns="host_location", inplace=True) 

# host verification derived:
verif_lists = df_airbnb_f["host_verifications"].apply(parse_list_string)
df_airbnb_f["host_verifications_count"] = verif_lists.apply(len)
df_airbnb_f["host_verif_email"] = verif_lists.apply(lambda x: int("email" in x))
df_airbnb_f["host_verif_phone"] = verif_lists.apply(lambda x: int("phone" in x))
df_airbnb_f["host_verif_work_email"] = verif_lists.apply(lambda x: int("work_email" in x))
df_airbnb_f.drop(columns="host_verifications", inplace=True) 

# bathroom derived:
s = df_airbnb_f["bathrooms_text"].astype(str).str.lower()

df_airbnb_f["bathroom_shared"] = s.str.contains("shared", na=False).astype(int)
df_airbnb_f["bathroom_private"] = s.str.contains("private", na=False).astype(int)
df_airbnb_f["bathroom_half"] = s.str.contains("half-bath", na=False).astype(int)

extracted = s.str.extract(r"(\d+(\.\d+)?)")[0].astype(float)
df_airbnb_f["nr_bathrooms"] = df_airbnb_f["bathrooms"].fillna(extracted)
df_airbnb_f.drop(columns="bathrooms_text", inplace=True) 

# neighbourhood_filtered
# invalid_neighbourhoods = ["Rathaus","Gewerbeschule", "Hochschulen", "City"] # actually real qaurtiere, no need to filter
# df_airbnb_f["neighbourhood_cleansed"] = df_airbnb_f["neighbourhood_cleansed"].replace(invalid_neighbourhoods , pd.NA)

# text columns
text_cols = ["host_country", "host_location", "host_response_time", "neighbourhood_cleansed","neighbourhood_group_cleansed", "room_type", "property_type"]
for col in text_cols:
    if col in df_airbnb_f.columns:
        df_airbnb_f[col] = df_airbnb_f[col].astype("string")

# drop amenities as too noisy
amenities_df = df_airbnb_f[["amenities"]].copy()
amenities_df["amenities"].apply(parse_list_string)
df_airbnb_f.drop(columns="amenities", inplace=True) 

# handle categorical:
cat_cols = [
    "host_country",
    "neighbourhood_cleansed",
    "neighbourhood_group_cleansed",
    "room_type",
    "property_type"
]

for col in cat_cols:
    if col in df_airbnb_f.columns:
        df_airbnb_f[col] = df_airbnb_f[col].astype("category")

response_order = [
    "within an hour",
    "within a few hours",
    "within a day",
    "a few days or more"
]

df_airbnb_f["host_response_time"] = pd.Categorical(
    df_airbnb_f["host_response_time"],
    categories=response_order,
    ordered=True
)

In [ ]:
df_airbnb_f

# Process amenities

In [ ]:
from collections import Counter
import pandas as pd

# Parse once
amenity_lists = amenities_df["amenities"].apply(parse_list_string)
amenity_lists_norm = amenity_lists.apply(
    lambda items: [str(x).strip().lower() for x in items]
)

# Print most common exact amenities
amenity_counter = Counter()
for items in amenity_lists_norm:
    amenity_counter.update(set(items))   # count per listing, not duplicates within listing

top_n = 500
amenity_freq = pd.DataFrame(
    amenity_counter.items(),
    columns=["amenity", "listing_count"]
).sort_values("listing_count", ascending=False)

amenity_freq["share"] = amenity_freq["listing_count"] / len(df_airbnb_f)

print("\nMost common exact amenities:")
print(amenity_freq.head(top_n).to_string(index=False))

In [ ]:
# look for variations to correctly filter
word="parking"

print([i for i in amenity_freq["amenity"] if word in i])

In [ ]:
def has_wifi(items):
    return bool(any("wifi" in item for item in items))

def has_tv(items):
    return bool(any(("tv" in item) or ("hdtv" in item) for item in items))

def has_amenities_df(items):
    return bool(any("amenities_df" in item for item in items))

def has_air_conditioning(items):
    
    return bool(any("air conditioning" in item for item in items))

def has_laundry(items):
    return bool(any(
        (
            (("washer" in item) and ("dishwasher" not in item))
            or
            (("dryer" in item) and ("hair dryer" not in item))
        )
        for item in items
    ))

def has_free_parking(items):
    return bool(any(("parking" in item) and ("free" in item) for item in items))

def has_paid_parking(items):
    return bool(any(("parking" in item) and ("paid" in item) for item in items))

def has_housekeeping(items):
    return bool(any("housekeeping" in item for item in items))

def has_kitchen(items):
    return bool(any(item == "kitchen" for item in items))

def has_iron(items):
    return bool(any(item == "iron" for item in items))

def has_dishwasher(items):
    return bool(any("dishwasher" in item for item in items))

def has_elevator(items):
    return bool(any("elevator" in item for item in items))

def has_pets_allowed(items):
    return bool(any("pets allowed" in item for item in items))

def has_bathtub(items):
    return bool(any("bathtub" in item for item in items))

def has_refrigerator(items):
    return bool(any("refrigerator" in item or "fridge" in item for item in items))

def has_oven(items):
    return bool(any("oven" in item for item in items))

def has_stove(items):
    return bool(any("stove" in item for item in items))

def has_freezer(items):
    return bool(any("freezer" in item for item in items))

def has_microwave(items):
    return bool(any("microwave" in item for item in items))

def has_self_checkin(items):
    return bool(any(
        ("self check-in" in item) or ("lockbox" in item) or ("keypad" in item) or ("smart lock" in item)
        for item in items
    ))

def has_toaster(items):
    return bool(any("toaster" in item for item in items))

def has_outdoor_dining(items):
    return bool(any("outdoor dining area" in item for item in items))

def has_outdoor_space(items):
    return bool(any(
        ("patio" in item) or ("balcony" in item) or ("backyard" in item)
        for item in items
    ))

def has_blender(items):
    return bool(any("blender" in item for item in items))

def has_lake_access(items):
    return bool(any("lake access" in item for item in items))

def has_rice_maker(items):
    return bool(any("rice maker" in item for item in items))

def has_board_games(items):
    return bool(any("board games" in item for item in items))

def has_city_view(items):
    return bool(any("city skyline view" in item for item in items))

def has_coffee_maker(items):
    return bool(any("coffee maker" in item for item in items))

def has_workspace(items):
    return bool(any("dedicated workspace" in item for item in items))

def has_hair_dryer(items):
    return bool(any("hair dryer" in item for item in items))

def has_heating(items):
    return bool(any("heating" in item for item in items))
amenities_df["amenities_count"] = amenity_lists_norm.apply(len)

amenities_df["amenity_wifi"] = amenity_lists_norm.apply(has_wifi)
amenities_df["amenity_tv"] = amenity_lists_norm.apply(has_tv)
amenities_df["amenity_air_conditioning"] = amenity_lists_norm.apply(has_air_conditioning)
amenities_df["amenity_laundry"] = amenity_lists_norm.apply(has_laundry)
amenities_df["amenity_free_parking"] = amenity_lists_norm.apply(has_free_parking)
amenities_df["amenity_paid_parking"] = amenity_lists_norm.apply(has_paid_parking)
amenities_df["amenity_iron"] = amenity_lists_norm.apply(has_iron)
amenities_df["amenity_housekeeping"] = amenity_lists_norm.apply(has_housekeeping)

amenities_df["amenity_kitchen"] = amenity_lists_norm.apply(has_kitchen)
amenities_df["amenity_dishwasher"] = amenity_lists_norm.apply(has_dishwasher)
amenities_df["amenity_elevator"] = amenity_lists_norm.apply(has_elevator)
amenities_df["amenity_pets_allowed"] = amenity_lists_norm.apply(has_pets_allowed)

amenities_df["amenity_bathtub"] = amenity_lists_norm.apply(has_bathtub)
amenities_df["amenity_refrigerator"] = amenity_lists_norm.apply(has_refrigerator)
amenities_df["amenity_oven"] = amenity_lists_norm.apply(has_oven)
amenities_df["amenity_stove"] = amenity_lists_norm.apply(has_stove)
amenities_df["amenity_freezer"] = amenity_lists_norm.apply(has_freezer)
amenities_df["amenity_microwave"] = amenity_lists_norm.apply(has_microwave)

amenities_df["amenity_self_checkin"] = amenity_lists_norm.apply(has_self_checkin)
amenities_df["amenity_toaster"] = amenity_lists_norm.apply(has_toaster)
amenities_df["amenity_outdoor_dining"] = amenity_lists_norm.apply(has_outdoor_dining)
amenities_df["amenity_outdoor_space"] = amenity_lists_norm.apply(has_outdoor_space)

amenities_df["amenity_blender"] = amenity_lists_norm.apply(has_blender)
amenities_df["amenity_lake_access"] = amenity_lists_norm.apply(has_lake_access)
amenities_df["amenity_rice_maker"] = amenity_lists_norm.apply(has_rice_maker)
amenities_df["amenity_board_games"] = amenity_lists_norm.apply(has_board_games)
amenities_df["amenity_city_view"] = amenity_lists_norm.apply(has_city_view)

amenities_df["amenity_coffee_maker"] = amenity_lists_norm.apply(has_coffee_maker)
amenities_df["amenity_workspace"] = amenity_lists_norm.apply(has_workspace)
amenities_df["amenity_hair_dryer"] = amenity_lists_norm.apply(has_hair_dryer)
amenities_df["amenity_heating"] = amenity_lists_norm.apply(has_heating)


# Some extra ones one might want to filter out:


def has_backyard(items):
    return bool(any("backyard" in item for item in items))

def has_water_access(items):
    return bool(any(
        ("lake access" in item) or
        ("waterfront" in item) or
        ("beach access" in item) or
        ("shared beach access" in item)
        for item in items
    ))

def has_water_view(items):
    return bool(any(
        ("lake view" in item) or
        ("river view" in item) or
        ("waterfront" in item) or
        ("canal view" in item) or
        ("sea view" in item)
        for item in items
    ))

def has_mountain_view(items):
    return bool(any("mountain view" in item for item in items))

def has_city_view(items):
    return bool(any("city skyline view" in item for item in items))

def has_safe(items):
    return bool(any(item == "safe" for item in items))

def has_sound_system(items):
    return bool(any("sound system" in item for item in items))

def has_fireplace(items):
    return bool(any("fireplace" in item for item in items))

def has_exercise_equipment(items):
    return bool(any("exercise equipment" in item for item in items))

def has_ev_charger(items):
    return bool(any("ev charger" in item for item in items))

def has_family_items(items):
    return bool(any(
        ("crib" in item) or
        ("high chair" in item) or
        ("baby bath" in item) or
        ("changing table" in item) or
        ("children’s books and toys" in item) or
        ("pack ’n play" in item)
        for item in items
    ))

amenities_df["amenity_backyard"] = amenity_lists_norm.apply(has_backyard)
amenities_df["amenity_water_access"] = amenity_lists_norm.apply(has_water_access)
amenities_df["amenity_water_view"] = amenity_lists_norm.apply(has_water_view)
amenities_df["amenity_mountain_view"] = amenity_lists_norm.apply(has_mountain_view)
amenities_df["amenity_city_view"] = amenity_lists_norm.apply(has_city_view)
amenities_df["amenity_safe"] = amenity_lists_norm.apply(has_safe)
amenities_df["amenity_sound_system"] = amenity_lists_norm.apply(has_sound_system)
amenities_df["amenity_fireplace"] = amenity_lists_norm.apply(has_fireplace)
amenities_df["amenity_exercise_equipment"] = amenity_lists_norm.apply(has_exercise_equipment)
amenities_df["amenity_ev_charger"] = amenity_lists_norm.apply(has_ev_charger)
amenities_df["amenity_family_items"] = amenity_lists_norm.apply(has_family_items)

In [ ]:
overview(amenities_df)

In [ ]:
cols_to_add = amenities_df.drop(columns="amenities", errors="ignore").columns
print(cols_to_add.tolist())

df_airbnb_f = pd.concat([df_airbnb_f, amenities_df[cols_to_add]], axis=1)

# Housing

In [ ]:
housing_df.head()

In [ ]:
to_drop = ["datenstandcd"]

housing_df_f = housing_df.copy()
housing_df_f = housing_df_f.drop(columns=to_drop)

# handle categorical:
cat_cols = [
    "anzzimmerlevel2lang_nodm",
    "eigentuemersszpubl1lang",
    "kreislang",
    "room_type",
    "property_type",
    "quarlang"
]

for col in cat_cols:
    if col in housing_df_f.columns:
        housing_df_f[col] = housing_df_f[col].astype("category")

for col in housing_df_f.columns:
    print(col, housing_df_f[col].dtype)
    print(housing_df_f[col].unique()[:50], "\n")

# Rental

In [ ]:
rental_df.head()

In [ ]:
import pandas as pd

rental_df_f = rental_df.copy()

# =========================
# Time
# =========================
rental_df_f["stichtagdatmonat"] = pd.to_datetime(
    rental_df_f["stichtagdatmonat"].astype(str).str.replace(".", "-", regex=False),
    format="%Y-%m",
    errors="coerce"
)

rental_df_f["is_april_2024"] = (rental_df_f["stichtagdatjahr"] == 2024).astype("bool")

rental_df_f = rental_df_f.drop(
    columns=["stichtagdatjahr", "stichtagdatmonat"],
    errors="ignore"
)

# =========================
# raumeinheit
# keep label, drop code
# =========================
if "raumeinheitlang" in rental_df_f.columns:
    rental_df_f["raumeinheitlang"] = rental_df_f["raumeinheitlang"].astype("category")

rental_df_f = rental_df_f.drop(columns=["raumeinheitsort"], errors="ignore")

# =========================
# gemeinnuetzig
# derive bool, drop raw columns
# =========================
rental_df_f["is_gemeinnuetzig"] = rental_df_f["gemeinnuetziglang"].map({
    "Gemeinnützig": True,
    "Nicht gemeinnützig": False
}).astype("bool")

rental_df_f = rental_df_f.drop(
    columns=["gemeinnuetzigsort", "gemeinnuetziglang"],
    errors="ignore"
)

# =========================
# einheit
# derive bool, drop raw columns
# =========================
rental_df_f["is_per_sqm"] = rental_df_f["einheitlang"].map({
    "Quadratmeter": True,
    "Wohnung": False
}).astype("bool")

rental_df_f = rental_df_f.drop(
    columns=["einheitsort", "einheitlang"],
    errors="ignore"
)

# =========================
# preisart
# derive bool, drop raw columns
# =========================
rental_df_f["is_brutto"] = rental_df_f["preisartlang"].map({
    "brutto": True,
    "netto": False
}).astype("bool")

rental_df_f = rental_df_f.drop(
    columns=["preisartsort", "preisartlang"],
    errors="ignore"
)

# =========================
# zimmer
# clean labels, set mixed bucket to missing, make ordered category
# =========================
if "zimmerlang" in rental_df_f.columns:
    rental_df_f["zimmerlang"] = (
        rental_df_f["zimmerlang"]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({
            "2 , 3 und 4 Zimmer": pd.NA,
            "2 , 3  und 4 Zimmer": pd.NA
        })
    )

    room_order = ["2 Zimmer", "3 Zimmer", "4 Zimmer"]
    rental_df_f["zimmerlang"] = pd.Categorical(
        rental_df_f["zimmerlang"],
        categories=room_order,
        ordered=True
    )

rental_df_f = rental_df_f.drop(columns=["zimmersort"], errors="ignore")

# =========================
# gliederung normalization
# =========================
def normalize_gliederung(x):
    if pd.isna(x):
        return pd.Series({
            "gliederung_type": pd.NA,
            "gliederung_value": pd.NA,
            "gliederung_kreis": pd.NA,
            "gliederung_quartier": pd.NA,
            "gliederung_aggregated_area": pd.NA,
            "gliederung_contract_age": pd.NA,
            "is_city": pd.NA,
        })

    x = str(x).strip()

    if x == "Ganze Stadt":
        return pd.Series({
            "gliederung_type": "city",
            "gliederung_value": x,
            "gliederung_kreis": pd.NA,
            "gliederung_quartier": pd.NA,
            "gliederung_aggregated_area": pd.NA,
            "gliederung_contract_age": pd.NA,
            "is_city": True,
        })

    if any(k in x for k in ["Neubau", "Neubezug", "Mietverträge"]):
        return pd.Series({
            "gliederung_type": "contract_age",
            "gliederung_value": x,
            "gliederung_kreis": pd.NA,
            "gliederung_quartier": pd.NA,
            "gliederung_aggregated_area": pd.NA,
            "gliederung_contract_age": x,
            "is_city": False,
        })

    if x.startswith("Kreis "):
        return pd.Series({
            "gliederung_type": "kreis",
            "gliederung_value": x,
            "gliederung_kreis": x,
            "gliederung_quartier": pd.NA,
            "gliederung_aggregated_area": pd.NA,
            "gliederung_contract_age": pd.NA,
            "is_city": False,
        })

    if "(Kreis " in x:
        return pd.Series({
            "gliederung_type": "aggregated_area",
            "gliederung_value": x,
            "gliederung_kreis": pd.NA,
            "gliederung_quartier": pd.NA,
            "gliederung_aggregated_area": x,
            "gliederung_contract_age": pd.NA,
            "is_city": False,
        })

    return pd.Series({
        "gliederung_type": "quartier",
        "gliederung_value": x,
        "gliederung_kreis": pd.NA,
        "gliederung_quartier": x,
        "gliederung_aggregated_area": pd.NA,
        "gliederung_contract_age": pd.NA,
        "is_city": False,
    })

if "gliederunglang" in rental_df_f.columns:
    gliederung_norm = rental_df_f["gliederunglang"].apply(normalize_gliederung)
    rental_df_f = pd.concat([rental_df_f, gliederung_norm], axis=1)

rental_df_f = rental_df_f.drop(
    columns=["gliederungsort", "gliederunglang"],
    errors="ignore"
)

# =========================
# categories for normalized columns
# =========================
cat_cols = [
    "raumeinheitlang",
    "zimmerlang",
    "gliederung_type",
    "gliederung_value",
    "gliederung_kreis",
    "gliederung_quartier",
    "gliederung_aggregated_area",
    "gliederung_contract_age",
]

for col in cat_cols:
    if col in rental_df_f.columns:
        rental_df_f[col] = rental_df_f[col].astype("category")

# =========================
# inspect result
# =========================
for col in rental_df_f.columns:
    print(col, rental_df_f[col].dtype)
    print(rental_df_f[col].unique()[:50], "\n")

In [ ]:
import pandas as pd

rental_df_f = rental_df.copy()

# Time
rental_df_f["stichtagdatmonat"] = pd.to_datetime(
    rental_df_f["stichtagdatmonat"].astype(str).str.replace(".", "-", regex=False),
    format="%Y-%m",
    errors="coerce"
)

rental_df_f["is_april_2024"] = (rental_df_f["stichtagdatjahr"] == 2024).astype("bool")

rental_df_f = rental_df_f.drop(
    columns=["stichtagdatjahr", "stichtagdatmonat"],
    errors="ignore"
)

# raumeinheit
if "raumeinheitlang" in rental_df_f.columns:
    rental_df_f["raumeinheitlang"] = rental_df_f["raumeinheitlang"].astype("category")

rental_df_f = rental_df_f.drop(columns=["raumeinheitsort"], errors="ignore")


# gemeinnuetzig
rental_df_f["is_gemeinnuetzig"] = rental_df_f["gemeinnuetziglang"].map({
    "Gemeinnützig": True,
    "Nicht gemeinnützig": False
}).astype("bool")

rental_df_f = rental_df_f.drop(
    columns=["gemeinnuetzigsort", "gemeinnuetziglang"],
    errors="ignore"
)

# einheit
rental_df_f["is_per_sqm"] = rental_df_f["einheitlang"].map({
    "Quadratmeter": True,
    "Wohnung": False
}).astype("bool")

rental_df_f = rental_df_f.drop(
    columns=["einheitsort", "einheitlang"],
    errors="ignore"
)

# preisart

rental_df_f["is_brutto"] = rental_df_f["preisartlang"].map({
    "brutto": True,
    "netto": False
}).astype("bool")

rental_df_f = rental_df_f.drop(
    columns=["preisartsort", "preisartlang"],
    errors="ignore"
)

# zimmer
if "zimmerlang" in rental_df_f.columns:
    rental_df_f["zimmerlang"] = (
        rental_df_f["zimmerlang"]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({
            "2 , 3 und 4 Zimmer": pd.NA,
            "2 , 3  und 4 Zimmer": pd.NA
        })
    )

    room_order = ["2 Zimmer", "3 Zimmer", "4 Zimmer"]
    rental_df_f["zimmerlang"] = pd.Categorical(
        rental_df_f["zimmerlang"],
        categories=room_order,
        ordered=True
    )

rental_df_f = rental_df_f.drop(columns=["zimmersort"], errors="ignore")

# gliederung unpack
if "gliederunglang" in rental_df_f.columns:
    gl = rental_df_f["gliederunglang"].astype("string").str.strip()

    contract_mask = gl.str.contains("Neubau|Neubezug|Mietverträge", regex=True, na=False)
    kreis_mask = gl.str.startswith("Kreis ", na=False)
    area_mask = gl.str.contains(r"\(Kreis \d+\)", regex=True, na=False)
    city_mask = gl.eq("Ganze Stadt")

    rental_df_f["gliederung_city"] = gl.where(city_mask)
    rental_df_f["gliederung_kreis"] = gl.where(kreis_mask)
    rental_df_f["gliederung_area"] = gl.where(area_mask)
    rental_df_f["gliederung_contract_age"] = gl.where(contract_mask)
    rental_df_f["gliederung_quartier"] = gl.where(
        ~(city_mask | kreis_mask | area_mask | contract_mask)
    )

rental_df_f = rental_df_f.drop(
    columns=["gliederungsort", "gliederunglang"],
    errors="ignore"
)

# categories for unpacked columns
cat_cols = [
    "raumeinheitlang",
    "zimmerlang",
    "gliederung_city",
    "gliederung_kreis",
    "gliederung_area",
    "gliederung_contract_age",
    "gliederung_quartier",
]

for col in cat_cols:
    if col in rental_df_f.columns:
        rental_df_f[col] = rental_df_f[col].astype("category")

# =========================
# inspect result
# =========================
for col in rental_df_f.columns:
    print(col, rental_df_f[col].dtype)
    print(rental_df_f[col].unique()[:50], "\n")

In [ ]:
# is weird. "Neubau", "Neubezug" and "Affoltern" mean very different things. Maybe best to drop
for x in rental_df["gliederunglang"].unique():
    print(x)